In [2]:
%%capture
!pip install -q -U transformers accelerate bitsandbytes

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [5]:
%%capture
!pip install -q -U trl transformers accelerate peft datasets bitsandbytes

In [6]:
def analyze_threat(log_data):
    # Modele örnekler vererek "Düşünce Zinciri" oluşturuyoruz (Madde 7)
    prompt = f"""
    Sen bir Kıdemli Siber Güvenlik Analistisin. Aşağıdaki log verisini analiz et.

    Adım 1: Olağandışı aktiviteleri belirle.
    Adım 2: Potansiyel saldırı tipini (SQL Injection, Brute Force vb.) teşhis et.
    Adım 3: Acil eylem planı öner.

    LOG VERİSİ: {log_data}
    ANALİZ:
    """

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=250, temperature=0.4)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).split("ANALİZ:")[-1]

# Test: Şüpheli bir giriş denemesi logu
log_ornegi = "IP: 192.168.1.45 - 500 failed login attempts in 2 minutes for user 'admin'"
print(analyze_threat(log_ornegi))

In [ ]:
# 1. Gerekli Kütüphaneleri Yükle
!pip install -q -U transformers accelerate bitsandbytes gradio matplotlib

import torch
import gradio as gr
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import io
from PIL import Image

# 2. Model ve Bellek Yapılandırması (RAM Koruma)
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")

# 3. İstatistiksel Görselleştirme Fonksiyonu
def create_stat_plot(threat_counts):
    plt.figure(figsize=(6, 4))
    plt.bar(threat_counts.keys(), threat_counts.values(), color=['red', 'orange', 'blue'])
    plt.title("Tespit Edilen Tehdit Dağılımı")
    plt.ylabel("Adet")

    buf = io.BytesIO()
    plt.savefig(buf, format='png')
    buf.seek(0)
    return Image.open(buf)

# Saldırı sayacı (Görselleştirme için)
stats = {"Kritik": 0, "Orta": 0, "Düşük": 0}

# 4. Analiz ve Arayüz Fonksiyonu
def analyze_and_display(log_data):
    # Chain of Thought Prompt
    prompt = f"Analist Modu: AKTİF\nSistem Logu: {log_data}\n\nAnaliz Adımları:\n1. Tehdit Seviyesi Belirle\n2. Saldırı Tipini Tanımla\n3. Çözüm Önerisi Sun\n\nSONUÇ:"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.3)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True).split("SONUÇ:")[-1]

    # Basit bir mantıkla istatistik güncelleme
    if "kritik" in response.lower() or "saldırı" in response.lower():
        stats["Kritik"] += 1
    elif "şüpheli" in response.lower():
        stats["Orta"] += 1
    else:
        stats["Düşük"] += 1

    return response, create_stat_plot(stats)

# 5. Gradio Web Arayüzü Tasarımı
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🛡️ Cyber-LLM: Tehdit Analiz Platformu")
    gr.Markdown("Log verilerini girin, Yapay Zeka saniyeler içinde analiz etsin.")

    with gr.Row():
        with gr.Column():
            log_input = gr.Textbox(label="Sistem Logu / Ağ Verisi", placeholder="Örn: 192.168.1.1 üzerinden SQL injection denemesi...")
            submit_btn = gr.Button("Analiz Et", variant="primary")

        with gr.Column():
            output_text = gr.Textbox(label="AI Analiz Raporu")
            output_plot = gr.Image(label="Tehdit İstatistikleri")

    submit_btn.click(analyze_and_display, inputs=log_input, outputs=[output_text, output_plot])

# Arayüzü Başlat
demo.launch(debug=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 145.6 MB/s eta 0:00:00


/tmp/ipython-input-2224463567.py:57: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://21316b8021eabfb328.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
